# QCQP Forward

Train a surrogate model with a quadratic inequality in the symbolic problem.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from kkthn import KKTHardNet

TRAIN = {
    "epochs": 2,
    "batch_size": 8,
    "learning_rate": 1e-3,
    "train_frac": 0.8,
    "hidden_size": 16,
    "hidden_layers": 1,
    "seed": 42,
    "dtype": "float64",
    "print_every": 1,
}


## Generate Example Data

In [ ]:
data_dir = Path('notebooks/data/qcqp_forward')
data_dir.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(1)
x1 = rng.uniform(0.2, 1.2, size=32)
x2 = rng.uniform(-0.3, 0.3, size=32)
y1 = 0.6 * x1
y2 = x2

pd.DataFrame({'x1': x1, 'x2': x2}).to_csv(data_dir / 'parameters.csv', index=False)
pd.DataFrame({'y1': y1, 'y2': y2}).to_csv(data_dir / 'variables.csv', index=False)
data_dir


## Build And Train

In [ ]:
model = KKTHardNet(name='qcqp_forward', train=TRAIN)
x = model.add_parameter(['x1', 'x2'])
y = model.add_variable(['y1', 'y2'])
model.constraints.add(
    y.y1 - 0.6 * x.x1 == 0,
    y.y2 - x.x2 == 0,
    y.y1**2 + y.y2**2 <= 1.0,
)
model.dataset(parameters=data_dir / 'parameters.csv', variables=data_dir / 'variables.csv')
result = model.model()
result['metadata_path']


In [ ]:
KKTHardNet().load(result['metadata_path']).predict([0.9, 0.1])
